In [144]:
import os
import sys
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import json

sys.path.append(os.path.abspath("../.."))
from scraper import fetch_website_contents, fetch_website_links

In [135]:
openai = OpenAI(
    base_url="http://localhost:11500/v1",
    api_key="ollama"
)

MODEL = "llama3.2:3b"

In [105]:
links = fetch_website_links("https://ollama.com")
links

['/',
 '/search',
 '/docs',
 '/pricing',
 '/signin',
 '/download',
 '/search',
 '/download',
 '/docs',
 '/pricing',
 '/signin',
 'https://docs.ollama.com/integrations/openclaw',
 '/download/windows',
 '/download',
 'https://docs.ollama.com/integrations',
 '/signup',
 '/upgrade?plan=pro&interval=year',
 '/upgrade?plan=pro',
 '/upgrade?plan=max',
 '/download',
 '/download',
 '/blog',
 'https://docs.ollama.com',
 'https://github.com/ollama/ollama',
 'https://discord.com/invite/ollama',
 'https://twitter.com/ollama',
 'mailto:hello@ollama.com',
 '/privacy',
 '/terms',
 '/blog',
 '/download',
 'https://docs.ollama.com',
 'https://github.com/ollama/ollama',
 'https://discord.com/invite/ollama',
 'https://twitter.com/ollama',
 'https://lu.ma/ollama',
 '/privacy',
 '/terms']

## First we are using llama3.2:3b model to figure out which links are relevant for broucher.

In [126]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [127]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the working URL starting with https only in JSON format as per the system prompt instructions.,
Do not include Terms of Service, Privacy, email links, not found, empty links.

Include full https links only, do not include relative links or links that are not working.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [108]:
print(get_links_user_prompt("https://ollama.com"))


Here is the list of links on the website https://ollama.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the working URL starting with https only in JSON format as per the system prompt instructions.,
Do not include Terms of Service, Privacy, email links, not found, empty links.

Include full https links only, do not include relative links or links that are not working.

Links (some might be relative links):

/
/search
/docs
/pricing
/signin
/download
/search
/download
/docs
/pricing
/signin
https://docs.ollama.com/integrations/openclaw
/download/windows
/download
https://docs.ollama.com/integrations
/signup
/upgrade?plan=pro&interval=year
/upgrade?plan=pro
/upgrade?plan=max
/download
/download
/blog
https://docs.ollama.com
https://github.com/ollama/ollama
https://discord.com/invite/ollama
https://twitter.com/ollama
mailto:hello@ollama.com
/privacy
/terms
/blog
/download
https://docs.ollama.com
https://github.com/ollama/ollama
h

In [109]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [110]:
select_relevant_links("https://huggingface.co/")

Selecting relevant links for https://huggingface.co/ by calling openchat:7b
Found 8 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'models page', 'url': 'https://huggingface.co/models'},
  {'type': 'docs page', 'url': 'https://huggingface.co/docs'},
  {'type': 'changelog page', 'url': 'https://huggingface.co/changelog'},
  {'type': 'endpoints page', 'url': 'https://endpoints.huggingface.co'}]}

In [119]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        if link['url'].startswith("https://"):
            result += f"\n\n### Link: {link['type']}\n"
            result += fetch_website_contents(link["url"])
    return result

In [120]:
print(fetch_page_and_all_relevant_links("https://huggingface.co/"))

Selecting relevant links for https://huggingface.co/ by calling openchat:7b
Found 3 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
SulphurAI/Sulphur-2-base
Updated
6 days ago
•
784k
•
941
openbmb/MiniCPM-V-4.6
Updated
about 24 hours ago
•
22.5k
•
540
HiDream-ai/HiDream-O1-Image
Updated
32 minutes ago
•
11.7k
•
328
Zyphra/ZAYA1-8B
Updated
3 days ago
•
141k
•
494
deepseek-ai/DeepSeek-V4-Pro
Updated
9 days ago
•
2.77M
•
3.96k
Browse 2M+ models
Spaces
Running
on
Zero
MCP
1.16k
Wan2.2 14B Fast Preview
🐌
1.16k
generate a video from an image with a text prompt
Running
on
Zero
Agents
Featured
98
Pixal3D
🏆
98
High-fidelity pixel-aligned image-to-3D generation.
Running

In [121]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

In [124]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:7_000] # Truncate if more than 7,000 characters
    return user_prompt

In [138]:
get_brochure_user_prompt("edwarddonner", "https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2:3b
Found 1 relevant links


'\nYou are looking at a company called: edwarddonner\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHome - Edward Donner\n\nHome\nAI Curriculum\nProficient AI Engineer\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,\nacquired in 2021\n.\nI will hap

In [133]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="llama3.2:3b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [130]:
create_brochure("HuggingFace", "https://huggingface.co/")

Selecting relevant links for https://huggingface.co/ by calling openchat:7b
Found 11 relevant links


 # Hugging Face Brochure

Hugging Face is the AI community building the future of machine learning. Our platform collaborates on models, datasets, and applications to create a rich ecosystem for researchers and developers alike. We empower creators with an accessible and powerful environment to innovate and grow the field of artificial intelligence.

## About Us

Our vision at Hugging Face is to enable everyone to collaboratively build a better world with AI. We strive to break down barriers, democratize access to state-of-the-art models and datasets, and empower the next generation of AI talent through community engagement. Our tools and resources are utilized by researchers, developers, and organizations across various industries, such as healthcare, finance, manufacturing, government, and beyond.

## Our Technology Stack

At Hugging Face, we leverage the power of Transformers and natural language processing to build cutting-edge AI applications. Key components of our technology stack include:

- Models: We host a collection of 2 million+ pre-trained models that can be used for various tasks such as translation, summarization, and sentiment analysis.
- Datasets: With access to over 500k datasets, users can find the data they need to train, verify, or test their AI applications.
- Spaces: Our interactive environment allows users to run AI applications in real time with various predefined models and configurations.

## Collaboration and Community

We believe in fostering collaboration within our community. We provide tools and resources that make it easy to contribute and learn from one another, including:

- Forums: Our discussion platform where users can exchange ideas, ask questions, and collaborate on projects.
- GitHub: As a part of the Hugging Face organization, we contribute to open-source projects and provide tools for developers to build their own AI applications.
- Blog and Weekly Papers: We share insights, updates, and industry news through our weekly blog and papers, keeping the community informed and engaged.

## Careers and Opportunities

Hugging Face is always looking for passionate and driven individuals to join our team. We have various opportunities for engineers, data scientists, product managers, and other roles. Check out our [Jobs](https://huggingface.com/careers) page for the latest openings and follow us on social media to stay updated on our upcoming events and announcements.

Join the AI revolution at Hugging Face – together we can build a better future through collaboration, innovation, and cutting-edge technology.

In [145]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="llama3.2:3b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [146]:
stream_brochure("Edward Donner", "https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2:3b
Found 9 relevant links


**Welcome to Edward Donner**

At Edward Donner, we're revolutionizing the world of work by applying cutting-edge artificial intelligence (AI) to the age-old problem of finding the right talent. Our pioneering technology helps recruiters source, understand, engage, and manage top performers with unprecedented accuracy and speed.

**Meet Our Team**

Our co-founder and CTO, Ed, is a seasoned expert in AI and has a passion for using this power to make a positive impact on people's lives. With over 5 years of experience developing AI solutions, he's developed a proprietary model that matches people with roles that spark inspiration, motivation, and fulfillment.

**Our Mission**

Our long-term goal is to raise the level of human prosperity by helping people find their purpose and potential. We believe in harnessing the power of AI to create meaningful connections between humans and technology.

**What Are We Working On?**

At Edward Donner, we're focused on developing innovative solutions that make a tangible difference. Our patented model leverages Generative AI and machine learning to connect top talent with dream jobs, where people can thrive and reach their full potential.

**Stay Inspired**

Join our journey as we rewrite the boundaries of what's possible in the world of work. Follow us on social media or tune in to Ed's popular live events and talks on Agents and LLMs. Let's revolutionize the future of work together!

**Careers at Edward Donner**

We're a dynamic team of innovators, thinkers, and problem-solvers who share Ed's passion for creating meaningful impact through AI. If you're ready to join a visionary company that's shaping the future of work, explore our job openings today!